In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

# 1. Load the data
df = pd.read_csv('titanic.csv')
print("Original Data:")
print(df.head())

# 2. Drop unnecessary columns
df = df.drop(['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin', 'Embarked'], axis='columns')

# 3. Handle missing values (Age has some empty rows)
df['Age'] = df['Age'].fillna(df['Age'].median())

# 4. Convert text columns (Sex) into numbers
le_sex = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex']) # female = 0, male = 1

# Separate inputs (X) and target (y)
inputs = df.drop('Survived', axis='columns')
target = df['Survived']

print("\nCleaned Data ready for the tree:")
print(inputs.head())

Original Data:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   Na

In [5]:
# 5. Create and train the model
model = DecisionTreeClassifier()
model.fit(inputs, target)

# 6. Check the accuracy
print("Model Accuracy on training data:", model.score(inputs, target))

# 7. Predict survival for a few hypothetical passengers
# Let's predict for:
# A 22-year-old male (Pclass 3, Fare 7.25) -> likely died
# A 38-year-old female (Pclass 1, Fare 71.28) -> likely survived
predictions = model.predict([[3, 1, 22.0, 7.25], 
                             [1, 0, 38.0, 71.28]])
print("\nPredictions (1 = Survived, 0 = Died):", predictions)

Model Accuracy on training data: 0.9775533108866442

Predictions (1 = Survived, 0 = Died): [0 1]


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


In [6]:
from sklearn import tree

# Print the tree structure as text
text_representation = tree.export_text(model, feature_names=list(inputs.columns))
print(text_representation)


|--- Sex <= 0.50
|   |--- Pclass <= 2.50
|   |   |--- Age <= 2.50
|   |   |   |--- Pclass <= 1.50
|   |   |   |   |--- class: 0
|   |   |   |--- Pclass >  1.50
|   |   |   |   |--- class: 1
|   |   |--- Age >  2.50
|   |   |   |--- Fare <= 28.86
|   |   |   |   |--- Fare <= 28.23
|   |   |   |   |   |--- Age <= 56.00
|   |   |   |   |   |   |--- Age <= 23.50
|   |   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |   |--- Age >  23.50
|   |   |   |   |   |   |   |--- Age <= 27.50
|   |   |   |   |   |   |   |   |--- Age <= 25.50
|   |   |   |   |   |   |   |   |   |--- Fare <= 13.75
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |   |   |--- Fare >  13.75
|   |   |   |   |   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |   |   |   |--- Age >  25.50
|   |   |   |   |   |   |   |   |   |--- Fare <= 17.43
|   |   |   |   |   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |   |   |   |   |--- Fare >  17.43
|   |   |   |   |   |   |  

In [ ]:
import matplotlib.pyplot as plt
from sklearn import tree

# Set a large figure size so the tree isn't tiny
plt.figure(figsize=(18, 10))

# Draw the tree
tree.plot_tree(model, 
               feature_names=inputs.columns, 
               class_names=['Died', 'Survived'], 
               filled=True, 
               rounded=True, 
               fontsize=10)

plt.title("Titanic Decision Tree", fontsize=16)
plt.show()
